# Nakhlah: EdgeNeXt-Base, same fair test as ConvNeXt-Tiny, BioCLIP and the others (5 folds, with and without augmentation)
**Model:** EdgeNeXt-Base (timm `edgenext_base.in21k_ft_in1k`): a hybrid built for edge devices: convolution blocks for local details plus split-depth transposed attention for global context. Pre-trained on ImageNet-21k, then ImageNet-1k (~18M numbers, close to ConvNeXt-Nano's ~15M).

**Type:** hybrid (CNN + attention), designed for edge devices like the Jetson.

**Fair test: everything is the same as the other models except the model.**

| Same for every model | Setting |
|---|---|
| Split | **Run 5's frozen split**, read from `Nakhlah_ATM_combined_5folds`. If it doesn't match exactly, the notebook **stops**. |
| Test `same_places` | Train on 85% of all 3 places, test on the locked 15% (513 crops). Crop macro F1 so far (no aug / aug): BioCLIP 87.7% / 86.5%, **ConvNeXt-Nano 84.8% / 85.4%**, EfficientNetV2-S 82.5% / 83.6%, ConvNeXt-Tiny 82.5% / 83.0%, TinyViT-11M 82.0% / 82.0% |
| Test `new_farm_Madinah` | Train on Al Ammariyah + Al Thumamah, test on exactly the same 665 Madinah crops as Run 3. ConvNeXt-Tiny 58.6% / 60.1%, BioCLIP 60.4% / 60.1%, ConvNeXt-Nano 58.2% / 59.8%, TinyViT-11M 58.3% / 59.2% |
| Crops | 224×224 (the same for every model), 10% padding, boxes under 20 px skipped |
| Augmentation | Same random changes, training crops only; never on the test |
| Training | 5 epochs head (step 1e-3) + 10 epochs whole model (step 1e-4, same as the other CNN and hybrid models), class-weighted loss, same seed |
| 5 folds | 4 folds train, 1 fold checks; the 5 fold models vote on the test |
| Metrics | Accuracy, balanced accuracy, macro precision, macro recall, macro F1, crop vs whole photo, per class, confusion matrices |

**Check before trusting the results:** cells 3 and 5 should print **"✅ Same split as the frozen one"**. If they print "No frozen split found…", the Run 5 folder isn't in this Drive; the split is rebuilt with the same method (and should be identical), but confirm with the check cell.

**How to run:** Runtime → Change runtime type → **T4 GPU**. Paste your Roboflow key in cell 1. Then **Run all**. Both tests take about **1.5–3 hours** (20 trainings). If Colab stops, run all again: finished folds are skipped. To run only the same-places test, put a `#` in front of the `new_farm_Madinah` line in `TESTS` in cell 1.

In [ ]:
# ===== CELL 1: settings + connect Drive  (EDGENEXT-BASE: same split, 5 folds, with and without augmentation - same test as ConvNeXt-Tiny and BioCLIP) =====
!pip install -q roboflow pillow-heif albumentations timm
from google.colab import drive
drive.mount("/content/drive")
import os

ROBOFLOW_KEY = "PASTE_YOUR_ROBOFLOW_API_KEY_HERE"
WORKSPACE = "lubn0-icloud-com"
# site name: (Roboflow project id from the project's URL, version number you generated)
SITES = {"Ammariyah": ("alamariah", 1),
         "Thumamah": ("althumamah", 1)}     # same versions as before, so the split matches
OLD = ("combined-dataset-for-naklah", 3)      # the old dataset: only its MADINAH photos are used
CLASS_MAP = {}                                # only if cell 2 asks: {"your class name": "old class name"}

SAVE = "/content/drive/MyDrive/Nakhlah_EdgeNeXtBase_5folds"       # results of THIS notebook (new folder)
RUN5 = "/content/drive/MyDrive/Nakhlah_ATM_combined_5folds"        # your Run 5 folder: its frozen split is reused
FREEZE_DIR = RUN5 if os.path.exists(RUN5 + "/roles.csv") else SAVE  # use Run 5's split if it is there
SPLIT_FILE = FREEZE_DIR + "/split.csv"
N_FOLDS, SEED = 5, 42
TEST_FRACTION = 0.15                   # 15% of each place is the locked test (same as Run 5)
BLOCK = 10                             # photos with neighbouring IMG numbers (IMG_9190..9199) = same leaf group
SIZE = 224                             # crop size
PAD = 0.10                             # keep 10% extra around each box
MIN_PX = 20                            # skip tiny boxes (smaller than 20 px)
BATCH, ACCUM = 32, 1
HEAD_EPOCHS, FINETUNE_EPOCHS = 5, 10   # same as ConvNeXt-Tiny
HEAD_LR, FINETUNE_LR = 1e-3, 1e-4      # same as ConvNeXt-Tiny
MODEL = "edgenext_base"

# the two tests (remove one to save time)
TESTS = {"same_places": "train on 85% of all 3 places -> test on the locked 15% (same as Run 5)",
         "new_farm_Madinah": "train on Al Ammariyah + Al Thumamah -> test on ALL Madinah (never seen)"}
SETUPS = ["no_aug", "online_aug"]
# ConvNeXt-Tiny results on the same tests, to compare (crop macro F1 no aug / aug)
CONVNEXT = {"same_places": (0.825, 0.830),             # Run 5
            "new_farm_Madinah": (0.586, 0.601)}       # Run 3 (exactly the same Madinah test)
print("Settings ready | split taken from:", FREEZE_DIR)

In [ ]:
# ===== CELL 2: download your two projects (+ the old dataset) and label every photo =====
import os, glob, re, shutil, hashlib, random, collections
import numpy as np, pandas as pd, yaml
from roboflow import Roboflow
from pillow_heif import register_heif_opener
register_heif_opener()
rf = Roboflow(api_key=ROBOFLOW_KEY)
ws = rf.workspace(WORKSPACE)


def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def load(project, version, folder):
    shutil.rmtree(folder, ignore_errors=True)
    ws.project(project).version(version).download("yolov12", location=folder)
    names = yaml.safe_load(open(folder + "/data.yaml"))["names"]
    names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
    rows, empty = [], 0
    for img in glob.glob(folder + "/*/images/*"):
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        cls = [int(float(l.split()[0])) for l in open(lab) if l.strip()] if os.path.exists(lab) else []
        if not cls:
            empty += 1
            continue
        base = os.path.basename(img).split(".rf.")[0]
        rows.append(dict(img=img, name=base, cls=names[collections.Counter(cls).most_common(1)[0][0]]))
    print(f"{project} v{version}: {len(rows)} photos with boxes, {empty} without boxes (skipped), classes = {names}")
    return pd.DataFrame(rows)


parts = []
for site, (project, version) in SITES.items():
    d = load(project, version, f"/content/ds_{site}")
    d["site"] = site
    num = d.name.str.extract(r"(\d+)")[0]
    d["group"] = [f"{site}_{int(n) // BLOCK}" if isinstance(n, str) else f"{site}_{b}" for n, b in zip(num, d.name)]
    parts.append(d)
df = pd.concat(parts, ignore_index=True)

# one class list for both sites (same class written differently, e.g. "Healthy" / "healthy", is merged)
df["key"] = df.cls.map(norm)
NAMES = sorted(df.groupby("key").cls.first())
KEY2NAME = {norm(n): n for n in NAMES}
df["cls"] = df.key.map(KEY2NAME)
C = {n: i for i, n in enumerate(NAMES)}

# exact duplicate photos: keep one copy (same photo with different classes: removed)
df["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in df.img]
bad = df.groupby("md5").cls.nunique()
df = df[~df.md5.isin(bad[bad > 1].index)].drop_duplicates("md5").reset_index(drop=True)

print("\nPhotos per class and site:")
print(pd.crosstab(df.cls, df.site, margins=True).to_string())
print(f"\nLeaf groups: {df.group.nunique()} (blocks of {BLOCK} neighbouring photo numbers)")
print("Example names:", list(df.name.head(5)))

# the old dataset: only its Madinah photos are used, for TRAINING (Qassim photos are dropped in cell 5)
old = load(OLD[0], OLD[1], "/content/ds_old")
old["cls"] = old.cls.map(lambda c: CLASS_MAP.get(c, c))
old["key"] = old.cls.map(norm)
missing = sorted(set(old.key) - set(KEY2NAME))
old = old[old.key.isin(KEY2NAME)]
oldtest = [(p, C[KEY2NAME[k]]) for p, k in zip(old.img, old.key)]
print(f"\nOld dataset photos usable as an other-place test: {len(oldtest)}")
if missing:
    print("⚠️ These old class names have no match in your classes, so those photos are skipped:", missing,
          "\n   If they mean the same thing, add them to CLASS_MAP in cell 1, e.g. {'nutrientDeficiency': 'Deficiency'}")

In [ ]:
# ===== CELL 3: SPLIT Al Ammariyah + Al Thumamah: locked TEST (15%) + 5 leaf-grouped folds, then FREEZE it =====
# The same split as Run 5 (checked against Run 5's saved split). Madinah is split the same way in cell 5.
from sklearn.model_selection import StratifiedGroupKFold
os.makedirs(SAVE, exist_ok=True)
sg = StratifiedGroupKFold(n_splits=round(1 / TEST_FRACTION), shuffle=True, random_state=SEED)
strat = df.site + "_" + df.cls                       # keep both sites and all classes spread evenly
p_idx, t_idx = next(sg.split(df, strat, df.group))
df["part"] = "pool"
df.loc[t_idx, "part"] = "locked_test"
df["fold"] = -1
pool = df[df.part == "pool"]
sg = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, v) in enumerate(sg.split(pool, strat[pool.index], pool.group)):
    df.loc[pool.index[v], "fold"] = k

assert set(df[df.part == "pool"].group).isdisjoint(df[df.part == "locked_test"].group), "LEAK: group in pool and test"
assert (df[df.part == "pool"].groupby("group").fold.nunique() == 1).all(), "LEAK: group split across folds"
print("✅ No leakage: every leaf group is in one place only")

FROZEN = SPLIT_FILE
cols = ["site", "name", "cls", "part", "fold"]
if os.path.exists(FROZEN):
    old_split = pd.read_csv(FROZEN, dtype={"name": str})[cols]
    same = sorted(map(tuple, old_split.values.tolist())) == sorted(map(tuple, df[cols].values.tolist()))
    if not same:
        raise SystemExit("❌ STOP: the split CHANGED (did the Roboflow version change?). Nothing was trained.")
    print("✅ Same split as the frozen one")
else:
    os.makedirs(os.path.dirname(FROZEN), exist_ok=True)
    df[cols].to_csv(FROZEN, index=False)
    print("No frozen split found, so this new one was saved to", FROZEN)

print(f"\nPool (5 folds): {(df.part == 'pool').sum()} | Locked TEST (never trained on): {(df.part == 'locked_test').sum()}")
print(pd.crosstab([df.part, df.fold], df.cls).to_string())

In [ ]:
# ===== CELL 4: the EdgeNeXt-Base model + the same training recipe as ConvNeXt-Tiny + the metrics =====
import timm
import time, torch, torch.nn as nn, albumentations as A
from PIL import Image, ImageOps
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)
to_tensor = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])   # ImageNet colours (same as ConvNeXt-Tiny)

# augmentation: the same as every earlier run (1-4 random changes, new every epoch)
rotate = A.OneOf([A.Sequential([A.Transpose(p=1), A.HorizontalFlip(p=1)]),
                  A.Sequential([A.HorizontalFlip(p=1), A.VerticalFlip(p=1)]),
                  A.Sequential([A.Transpose(p=1), A.VerticalFlip(p=1)])], p=1)
changes = [A.HorizontalFlip(p=1), A.VerticalFlip(p=1), rotate,
           A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=1),
           A.HueSaturationValue(hue_shift_limit=0, sat_shift_limit=15, val_shift_limit=10, p=1)]
AUG = A.Compose([A.OneOf([A.SomeOf(changes, n=k, replace=False, p=1) for k in (1, 2, 3, 4)], p=0.75),
                 A.MotionBlur(blur_limit=5, p=0.1)])


def reseed(t, counter):
    counter[0] += 1
    t.set_random_seed(counter[0])
    for ch in getattr(t, "transforms", []):
        reseed(ch, counter)


def worker_init(_):
    reseed(AUG, [torch.initial_seed() % 2**31])


class Photos(Dataset):
    def __init__(self, items, aug=False, tf=None):
        self.items, self.aug, self.tf = items, aug, (tf or to_tensor)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        p, y = self.items[i][:2]
        im = ImageOps.exif_transpose(Image.open(p)).convert("RGB").resize((SIZE, SIZE))
        if self.aug:
            im = Image.fromarray(AUG(image=np.array(im))["image"])
        return self.tf(im), y


def build(arch="edgenext_base"):
    m = timm.create_model("edgenext_base.in21k_ft_in1k", pretrained=True, num_classes=len(NAMES))
    return m, m.get_classifier()


def train(items, seed, aug, arch="edgenext_base"):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    reseed(AUG, [seed])
    g = torch.Generator().manual_seed(seed)
    tl = DataLoader(Photos(items, aug), batch_size=BATCH, shuffle=True, num_workers=2, generator=g,
                    drop_last=True, worker_init_fn=worker_init)
    cnt = np.bincount([it[1] for it in items], minlength=len(NAMES))
    w = np.where(cnt > 0, len(items) / (len(NAMES) * np.maximum(cnt, 1)), 0.0)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device))
    model, head = build(arch)
    model = model.to(device)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    def run(n, lr, params):
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n)
        for _ in range(n):
            model.train()
            opt.zero_grad()
            for i, (x, y) in enumerate(tl):
                x, y = x.to(device), y.to(device)
                with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                    loss = loss_fn(model(x), y) / ACCUM
                scaler.scale(loss).backward()
                if (i + 1) % ACCUM == 0:
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad()
            sched.step()

    for prm in model.parameters():
        prm.requires_grad = False
    for prm in head.parameters():
        prm.requires_grad = True
    run(HEAD_EPOCHS, HEAD_LR, head.parameters())
    for prm in model.parameters():
        prm.requires_grad = True
    run(FINETUNE_EPOCHS, FINETUNE_LR, model.parameters())
    return model


def probs_of(model, items, tf=None):
    model.eval()
    out = []
    with torch.no_grad(), torch.autocast(device_type="cuda", enabled=(device == "cuda")):
        for x, _ in DataLoader(Photos(items, tf=tf), batch_size=BATCH, num_workers=2):
            out.append(torch.softmax(model(x.to(device)).float(), 1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(NAMES)))


def all_metrics(t, p):
    """accuracy, balanced accuracy, macro precision / recall / F1 (classes present in the truth)."""
    t, p = np.array(t), np.array(p)
    if len(t) == 0:
        return {}
    labels = sorted(set(t))
    pr, rc, f1, _ = precision_recall_fscore_support(t, p, labels=labels, zero_division=0)
    return dict(accuracy=float((t == p).mean()), balanced_acc=balanced_accuracy_score(t, p),
                macro_precision=pr.mean(), macro_recall=rc.mean(), macro_F1=f1.mean())


def to_photos(items, preds):
    """Whole-photo answer = the most common answer among the photo's crops."""
    by = collections.defaultdict(list)
    for it, p in zip(items, preds):
        by[it[4]].append(p)
    pids = list(by)
    return [main_of[i] for i in pids], [collections.Counter(by[i]).most_common(1)[0][0] for i in pids]


def ms_per_crop(model, n=50):
    """Speed: milliseconds for ONE crop (batch of 1) on this Colab GPU. Only for comparing models."""
    model = model.to(device).eval()
    x = torch.randn(1, 3, SIZE, SIZE, device=device)
    with torch.no_grad():
        for _ in range(5):
            model(x)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n):
            model(x)
        if device == "cuda":
            torch.cuda.synchronize()
    return 1000 * (time.time() - t0) / n


print("Recipe ready | classes:", NAMES)

In [ ]:
# ===== CELL 5: ALL 3 PLACES COMBINED. Each place: locked TEST (15%) + 5 folds for training (85%). Then FREEZE. =====
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import GroupKFold


def leaf_of(name):
    """Madinah names look like '12-3': the first number is the leaf, so all photos of leaf 12 stay together."""
    m = re.match(r"^[^0-9]*?(\d+)[._\- ](\d+)", str(name))
    return m.group(1) if m else str(name)


def grouped_folds(d, n, strat):
    """Leaf-grouped, class-balanced folds; falls back to group-only if a class is too small."""
    try:
        return list(StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=SEED).split(d, strat, d.group))
    except ValueError:
        return list(GroupKFold(n_splits=n).split(d, strat, d.group))


# Al Ammariyah + Al Thumamah: the split from cell 3 (pool = training with folds, locked = TEST)
farms = df.copy()
farms["source"] = farms.site
farms["side"] = np.where(farms.part == "pool", "train", "test")
farms.loc[farms.side == "test", "fold"] = -1

# Madinah: exact duplicate photos kept once (a photo saved under two classes is removed),
# then the SAME method: locked TEST (15%) + 5 leaf-grouped folds on the rest
mad = old[~old.name.str.upper().str.startswith("QD")].copy()
mad["cls"] = mad.key.map(KEY2NAME)
mad["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in mad.img]
bad = mad.groupby("md5").cls.nunique()
mad = mad[~mad.md5.isin(bad[bad > 1].index)].drop_duplicates("md5").reset_index(drop=True)
mad["group"] = "Madinah_" + mad.cls + "_" + mad.name.map(leaf_of)
_, t_idx = grouped_folds(mad, round(1 / TEST_FRACTION), mad.cls)[0]
mad["side"] = "train"
mad.loc[t_idx, "side"] = "test"
mad["fold"] = -1
mp = mad[mad.side == "train"]
for k, (_, v) in enumerate(grouped_folds(mp, N_FOLDS, mp.cls)):
    mad.loc[mp.index[v], "fold"] = k
mad["source"] = "Madinah"
assert set(mad[mad.side == "train"].group).isdisjoint(mad[mad.side == "test"].group), "LEAK: Madinah leaf in train and test"
assert (mad[mad.side == "train"].groupby("group").fold.nunique() == 1).all(), "LEAK: a Madinah leaf is split across folds"

cols = ["img", "name", "cls", "fold", "source", "side"]
roles = pd.concat([farms[cols], mad[cols]], ignore_index=True)
roles["fold"] = roles.fold.astype(int)
roles["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in roles.img]
assert not set(roles[roles.side == "train"].md5) & set(roles[roles.side == "test"].md5), "LEAK: same photo in training and test"
assert (roles[roles.side == "train"].fold.between(0, N_FOLDS - 1)).all(), "a training photo has no fold"
assert (roles[roles.side == "test"].fold == -1).all()
print("✅ No leakage: no leaf and no photo is in both training and the locked test; every training photo has one fold")

os.makedirs(SAVE, exist_ok=True)
FROZEN2 = FREEZE_DIR + "/roles.csv"          # Run 5's roles: must match exactly
fcols = ["source", "name", "cls", "side", "fold"]
if os.path.exists(FROZEN2):
    fz = pd.read_csv(FROZEN2, dtype={"name": str})[fcols]
    same = sorted(map(tuple, fz.astype(str).values.tolist())) == sorted(map(tuple, roles[fcols].astype(str).values.tolist()))
    if not same:
        raise SystemExit("❌ STOP: the split CHANGED. Nothing was trained.")
    print("✅ Same split as the frozen one")
else:
    roles[fcols].to_csv(FROZEN2, index=False)
    print("Saved the frozen split to", FROZEN2)
print("\nPhotos: training (5 folds) vs locked test, per place:")
print(pd.crosstab(roles.source, roles.side, margins=True).to_string())
print("\nTraining photos per fold:")
print(pd.crosstab(roles[roles.side == "train"].source, roles[roles.side == "train"].fold).to_string())

In [ ]:
# ===== CELL 6: BOX CROPS (each crop keeps its photo's source, fold and photo id) =====
CROP_SIZE = SIZE
CROPDIR = "/content/crops"
shutil.rmtree(CROPDIR, ignore_errors=True)
os.makedirs(CROPDIR)


def cut_boxes(img_path, folder, tag):
    names = yaml.safe_load(open(folder + "/data.yaml"))["names"]
    names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
    lab = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    im = ImageOps.exif_transpose(Image.open(img_path)).convert("RGB")
    W, H = im.size
    out = []
    for j, line in enumerate(l for l in open(lab) if l.strip()):
        c, xc, yc, w, h = line.split()[:5]
        key = norm(CLASS_MAP.get(names[int(float(c))], names[int(float(c))]))
        if key not in KEY2NAME:
            continue                                   # e.g. insect boxes (not one of your 4 classes)
        xc, yc, w, h = float(xc) * W, float(yc) * H, float(w) * W, float(h) * H
        if w < MIN_PX or h < MIN_PX:
            continue
        x0, y0 = max(0, xc - w / 2 - PAD * w), max(0, yc - h / 2 - PAD * h)
        x1, y1 = min(W, xc + w / 2 + PAD * w), min(H, yc + h / 2 + PAD * h)
        dst = f"{CROPDIR}/{tag}_{j}.jpg"
        im.crop((int(x0), int(y0), int(x1), int(y1))).resize((CROP_SIZE, CROP_SIZE)).save(dst, quality=95)
        out.append((dst, C[KEY2NAME[key]]))
    return out


folder_of = {s: f"/content/ds_{s}" for s in SITES}
folder_of["Madinah"] = "/content/ds_old"
crops = []                                     # (file, class, fold, source, photo id)
main_of = {}
for i, r in enumerate(roles.itertuples()):
    main_of[i] = C[r.cls]
    for dst, y in cut_boxes(r.img, folder_of[r.source], f"r{i}"):
        crops.append((dst, y, r.fold, r.source, i))
# the two tests, as lists of crops: (file, class, fold, place, photo id)
EXP = {"same_places": ([c for c in crops if c[2] >= 0],                                   # 85% of all 3 places, 5 folds
                       [c for c in crops if c[2] == -1]),                                 # locked 15% of all 3 places
       "new_farm_Madinah": ([c for c in crops if c[2] >= 0 and c[3] in ("Ammariyah", "Thumamah")],  # 85% of the 2 farms
                            [])}                                                           # filled just below

# new_farm_Madinah test = EXACTLY the Madinah test of ConvNeXt Run 3: every Madinah photo, as it was there
mad_run3 = old[~old.name.str.upper().str.startswith("QD")].reset_index(drop=True)
for j, r in enumerate(mad_run3.itertuples()):
    pid = len(roles) + j                                  # new photo ids, so they never mix with training photos
    main_of[pid] = C[KEY2NAME[r.key]]
    for dst, y in cut_boxes(r.img, "/content/ds_old", f"m3_{j}"):
        EXP["new_farm_Madinah"][1].append((dst, y, -1, "Madinah", pid))
print(f"Madinah test (same as ConvNeXt Run 3): {len(EXP['new_farm_Madinah'][1])} crops from {len(mad_run3)} photos (Run 3 had 665 crops, 484 photos)")
for e, (tr, te) in EXP.items():
    assert not {c[4] for c in tr} & {c[4] for c in te}, f"LEAK in {e}"
    print(f"{e}: train {len(tr)} crops | test {len(te)} crops | test places: {sorted({c[3] for c in te})}")

In [ ]:
# ===== CELL 7: TRAIN EdgeNeXt-Base: 5 folds x (no_aug, online_aug) for each test. Finished folds are skipped. =====
# Each fold model's answers are saved to Drive, so finished folds are skipped if Colab stops.
OUT = SAVE + "/predictions"
os.makedirs(OUT, exist_ok=True)
INFO = SAVE + "/model_info.csv"
info = pd.read_csv(INFO).to_dict("records") if os.path.exists(INFO) else []
VAL = SAVE + "/fold_checks.csv"
val_rows = pd.read_csv(VAL).to_dict("records") if os.path.exists(VAL) else []

for arch in [MODEL]:
    if not any(r["model"] == arch for r in info):
        m, _ = build(arch)
        info.append(dict(model=arch, kind="fine-tuned", params_M=sum(p.numel() for p in m.parameters()) / 1e6,
                         ms_per_crop=ms_per_crop(m)))
        pd.DataFrame(info).to_csv(INFO, index=False)
        del m
    for exp, (tr_all, te) in EXP.items():
        if exp not in TESTS:
            continue
        for setup in SETUPS:
            for k in range(N_FOLDS):
                base = f"{OUT}/{exp}__{arch}__{setup}__fold{k}"
                if os.path.exists(base + "_test.npy"):
                    print(f"skip {exp} | {arch} | {setup} | fold {k} (already done)")
                    continue
                tr = [c for c in tr_all if c[2] != k]
                va = [c for c in tr_all if c[2] == k]
                print(f"\n{exp} | {arch} | {setup} | fold {k}: train {len(tr)} crops | check {len(va)} crops")
                model = train(tr, SEED, aug=(setup == "online_aug"), arch=arch)   # same seed as the ConvNeXt runs
                pv = probs_of(model, va)
                np.save(base + "_test.npy", probs_of(model, te))
                f1 = all_metrics([c[1] for c in va], pv.argmax(1))["macro_F1"]
                val_rows.append(dict(test=exp, model=arch, setup=setup, fold=k, check_crop_macro_F1=f1))
                pd.DataFrame(val_rows).to_csv(VAL, index=False)
                print(f"   → fold check macro F1 (crops): {f1:.3f}")
                del model
                torch.cuda.empty_cache()
print("\n✅ EdgeNeXt-Base training done")

In [ ]:
# ===== CELL 8: RESULTS - the 5 EdgeNeXt-Base fold models vote on each test; every metric, crop vs whole photo =====
summary, per_class = [], []
for exp in TESTS:
    tr_all, te = EXP[exp]
    t = [c[1] for c in te]
    for setup in SETUPS:
        files = [f"{OUT}/{exp}__{MODEL}__{setup}__fold{k}_test.npy" for k in range(N_FOLDS)]
        if not all(os.path.exists(f) for f in files):
            print(f"(missing folds: {exp} | {setup} - run cell 7 again)")
            continue
        p = np.mean([np.load(f) for f in files], axis=0).argmax(1).tolist()
        pt, pp = to_photos(te, p)
        for level, (tt, ppp) in {"crop": (t, p), "whole photo": (pt, pp)}.items():
            summary.append(dict(test=exp, setup=setup, level=level, n=len(tt), **all_metrics(tt, ppp)))
            labels = sorted(set(tt))
            pr, rc, f1, sup = precision_recall_fscore_support(tt, ppp, labels=labels, zero_division=0)
            for i, c in enumerate(labels):
                per_class.append(dict(test=exp, setup=setup, level=level, cls=NAMES[c],
                                      precision=pr[i], recall=rc[i], F1=f1[i]))
            if level == "crop":
                print(f"\nConfusion matrix: {exp} | {setup} | crops (rows = real, columns = model's answer)")
                print(pd.DataFrame(confusion_matrix(tt, ppp, labels=range(len(NAMES))),
                                   index=["real " + n for n in NAMES], columns=["said " + n for n in NAMES]).to_string())

S = pd.DataFrame(summary)
PC = pd.DataFrame(per_class)
S.to_csv(SAVE + "/EDGENEXT_BASE_summary.csv", index=False)
PC.to_csv(SAVE + "/EDGENEXT_BASE_per_class.csv", index=False)
for exp, desc in TESTS.items():
    print(f"\n========== EDGENEXT-BASE | {exp.upper()}: {desc} ==========")
    print(S[S.test == exp].drop(columns="test").round(3).to_string(index=False))

print("\n========== PER CLASS (precision / recall / F1) ==========")
print(PC.pivot_table(index=["test", "level", "cls"], columns="setup", values=["precision", "recall", "F1"]).round(3).to_string())

print("\n========== EDGENEXT-BASE vs CONVNEXT-TINY (crop macro F1) ==========")
rows = []
for exp in TESTS:
    for j, setup in enumerate(SETUPS):
        b = S[(S.test == exp) & (S.setup == setup) & (S.level == "crop")]
        if len(b):
            rows.append(dict(test=exp, setup=setup, edgenext_base=round(b.macro_F1.iloc[0], 3), ConvNeXt_Tiny=CONVNEXT[exp][j],
                             difference=round(b.macro_F1.iloc[0] - CONVNEXT[exp][j], 3)))
print(pd.DataFrame(rows).to_string(index=False))
I = pd.DataFrame(info)
print(f"\nEdgeNeXt-Base size: {I.params_M.iloc[0]:.0f} million numbers (ConvNeXt-Tiny: 28) | "
      f"{I.ms_per_crop.iloc[0]:.1f} ms per crop on this Colab GPU (compare only; the Jetson is slower)")
V = pd.DataFrame(val_rows)
print("\nFold checks (crop macro F1, mean ± sd):")
print(V.groupby(["test", "setup"]).check_crop_macro_F1.agg(["mean", "std"]).round(3).to_string())